# 20 · Capstone: Benchmarking Pipeline

**Series:** LLM Systems & Inference (Finale)  
**Builds toward:** `benchpress` — rigorous LLM inference benchmark for Apple Silicon

In this capstone you will:
- Build an end-to-end benchmarking harness combining **speed** (TTFT, TBT, throughput) and **quality** (perplexity, task accuracy)
- Apply **bootstrap confidence intervals** to get statistically rigorous estimates instead of single-run point estimates
- Build a **multi-model comparison pipeline** that outputs a ranked table with significance tests
- Profile the **memory-bandwidth bottleneck** on Apple Silicon (M-series unified memory architecture)
- Understand what separates a rigorous benchmark from a "vibes check"

This ties together concepts from notebooks 12 (serving metrics), 13 (local models), and 19 (cost optimization).

In [ ]:
# Standard library + numpy/scipy/matplotlib + optional ollama client
%pip install -q numpy scipy matplotlib
# Ollama Python client (optional — degrades gracefully if not installed)
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("ollama not installed — using simulated timing data")

## Background

### Why Benchmarking LLMs is Hard

Most "benchmarks" you see online are vibes checks dressed up with bar charts:

- **Point estimates** — single run, no confidence intervals, no knowledge of whether differences are real
- **Fixed prompts** — no prompt diversity; one prompt suite may favor one model's tokenizer or system prompt format
- **Speed OR quality** — MLPerf measures throughput on datacenter hardware; MMLU measures accuracy but not speed
- **No cost analysis** — tokens per second without cost-per-token is only half the picture

The community has slowly gotten more rigorous:
- **lm-evaluation-harness** (EleutherAI, 2021+) — standardized task framework, used for most leaderboards
- **OpenLLM Leaderboard** (HuggingFace) — community benchmark aggregator
- **HELM** (Stanford, 2022) — holistic evaluation across multiple axes
- **MLCommons MLPerf Inference** — datacenter benchmarking with reproducibility requirements

But none of these are designed for **Apple Silicon consumer hardware** with **statistical rigor** on **combined speed+quality** axes.

### What `benchpress` Adds

1. **Statistical rigor**: Bootstrap CIs on every metric. "Model A is faster" only if the 95% CI on the difference excludes zero.
2. **Apple Silicon awareness**: Memory-bandwidth model calibrated to M-series chips, unified memory budget tracking.
3. **Combined ranking**: A single score that weights speed and quality with user-defined weights.
4. **Reproducibility**: Seed control, system state capture, run metadata in every output file.

### Statistical Foundation: Bootstrap Confidence Intervals

For a metric $\theta$ (e.g., median TTFT), we can't derive an analytic CI because the distribution is unknown. Bootstrap gives us an empirical CI:

1. Collect $n$ observations $x_1, \ldots, x_n$
2. Resample $B$ bootstrap samples of size $n$ **with replacement**
3. Compute $\hat{\theta}_b^*$ for each bootstrap sample
4. CI = percentiles of the $\hat{\theta}_b^*$ distribution

This makes no distributional assumptions and works for any statistic (mean, median, P99, ratio of means).

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import Optional, Callable
import time
import warnings
import sys
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Core measurement types ────────────────────────────────────────────────────

@dataclass
class RequestRecord:
    """Single request measurement record."""
    prompt_tokens: int
    generated_tokens: int
    ttft_ms: float           # time-to-first-token
    tbt_ms: float            # time-between-tokens (median inter-token)
    e2e_ms: float            # end-to-end latency
    generation_text: str     # the actual generated text
    quality_score: float     # 0–1 quality metric (perplexity-derived or task accuracy)

@dataclass 
class BenchmarkRun:
    """Results from benchmarking one model configuration."""
    model_name: str
    records: list[RequestRecord]
    system_info: dict
    
    @property
    def n(self): return len(self.records)
    
    def metric(self, fn: Callable, field: str) -> np.ndarray:
        return np.array([fn(r) for r in self.records])
    
    def ttft_ms(self) -> np.ndarray:
        return np.array([r.ttft_ms for r in self.records])
    
    def tbt_ms(self) -> np.ndarray:
        return np.array([r.tbt_ms for r in self.records])
    
    def tokens_per_second(self) -> np.ndarray:
        return np.array([r.generated_tokens / (r.e2e_ms / 1000) for r in self.records])
    
    def quality(self) -> np.ndarray:
        return np.array([r.quality_score for r in self.records])
    
    def cost_per_1k_tokens(self, cost_per_1k: float) -> np.ndarray:
        """Cost scaled by actual tokens generated."""
        return np.array([cost_per_1k * r.generated_tokens / 1000 for r in self.records])

print("Core data structures defined.")
print("RequestRecord fields:", [f for f in RequestRecord.__dataclass_fields__])


## Simulating Apple Silicon Inference Timing

Since we can't require all readers to have Ollama installed with specific models, we build a **calibrated simulation** based on the memory-bandwidth model from notebook 12, using real measurements from M-series chips.

The fundamental constraint on Apple Silicon is **memory bandwidth**:
- Each token decode requires loading all model weights from RAM
- Bandwidth (not compute) is the bottleneck for batch size 1

$$\text{TBT} = \frac{\text{model\_params} \times \text{bytes\_per\_param}}{\text{bandwidth\_GB/s}} \times \frac{1}{\text{efficiency}}$$

Prefill is compute-bound (parallel attention over all prompt tokens), so it scales with FLOP/token × sequence length.

In [ ]:
# ── Apple Silicon timing simulator ───────────────────────────────────────────

@dataclass
class AppleSiliconChip:
    name: str
    memory_bandwidth_gbs: float   # GB/s unified memory bandwidth
    max_ram_gb: int
    neural_engine_tops: float     # TOPS (used for rough compute estimate)

APPLE_CHIPS = {
    "M1":       AppleSiliconChip("M1",       68.25, 16,  11.0),
    "M1 Pro":   AppleSiliconChip("M1 Pro",   200.0, 32,  11.0),
    "M2":       AppleSiliconChip("M2",        100.0, 24,  15.8),
    "M2 Pro":   AppleSiliconChip("M2 Pro",   200.0, 32,  15.8),
    "M3":       AppleSiliconChip("M3",        100.0, 24,  18.0),
    "M3 Max":   AppleSiliconChip("M3 Max",   300.0, 128, 18.0),
    "M4":       AppleSiliconChip("M4",        120.0, 32,  38.0),
    "M4 Pro":   AppleSiliconChip("M4 Pro",   273.0, 64,  38.0),
    "M4 Max":   AppleSiliconChip("M4 Max",   546.0, 128, 38.0),
}

@dataclass
class ModelConfig:
    name: str
    param_billions: float
    quant_bits: int       # 4, 8, 16
    context_window: int
    
    @property
    def model_size_gb(self) -> float:
        bytes_per_param = self.quant_bits / 8
        return self.param_billions * 1e9 * bytes_per_param / 1e9
    
    @property
    def kv_cache_gb_per_token(self) -> float:
        # Rough: 2 * n_layers * d_model * 2 (k+v) * bytes / token
        # Approximation: ~0.5 MB per token per billion params (empirical)
        return self.param_billions * 0.5 / 1000  # GB

class AppleSiliconSimulator:
    """
    Calibrated timing simulator for Apple Silicon LLM inference.
    Derived from empirical measurements on M1–M4 chips.
    """
    
    def __init__(self, chip: AppleSiliconChip, model: ModelConfig,
                 efficiency: float = 0.72, noise_cv: float = 0.08):
        self.chip = chip
        self.model = model
        self.efficiency = efficiency
        self.noise_cv = noise_cv  # coefficient of variation for timing noise
        
        # Check if model fits
        if model.model_size_gb > chip.max_ram_gb * 0.85:
            raise ValueError(f"{model.name} ({model.model_size_gb:.1f} GB) doesn't fit "
                             f"on {chip.name} ({chip.max_ram_gb} GB RAM)")
    
    def prefill_latency_ms(self, n_tokens: int) -> float:
        """
        Prefill is compute-bound. FLOPs ≈ 2 * param_count * n_tokens
        Neural engine: param_count * 2 (forward pass) FLOPs.
        Simplified: use bandwidth model with parallelism bonus for long prompts.
        """
        # Empirical: ~1-3ms per token for 7B @ Q4, better for longer (batched prefill)
        base_ms_per_tok = (self.model.model_size_gb * 1e9 /
                           (self.chip.memory_bandwidth_gbs * 1e9 * self.efficiency * 1000))
        # Parallelism bonus: prefill processes all tokens at once
        parallelism_factor = max(0.2, 1.0 / np.log2(max(n_tokens, 2)))
        return base_ms_per_tok * n_tokens * parallelism_factor * 1000  # to ms
    
    def decode_latency_ms_per_token(self) -> float:
        """Decode is memory-bandwidth bound (autoregressive, batch=1)."""
        model_bytes = self.model.model_size_gb * 1e9
        bandwidth_bytes_per_sec = self.chip.memory_bandwidth_gbs * 1e9
        return model_bytes / (bandwidth_bytes_per_sec * self.efficiency) * 1000  # ms
    
    def simulate_request(self, prompt_tokens: int, output_tokens: int,
                          rng=None) -> RequestRecord:
        if rng is None: rng = np.random.default_rng()
        
        def noisy(x):
            return x * (1 + rng.normal(0, self.noise_cv))
        
        ttft = max(1.0, noisy(self.prefill_latency_ms(prompt_tokens)))
        tbt_base = self.decode_latency_ms_per_token()
        tbts = [max(0.5, noisy(tbt_base)) for _ in range(output_tokens)]
        tbt_median = np.median(tbts)
        e2e = ttft + sum(tbts)
        
        # Quality: simulate perplexity-derived score (lower ppl = higher score)
        # Larger models / lower quant = better quality
        base_quality = 0.65 + 0.04 * self.model.param_billions + 0.03 * self.model.quant_bits
        base_quality = min(0.97, base_quality)
        quality = float(np.clip(rng.normal(base_quality, 0.04), 0.0, 1.0))
        
        return RequestRecord(
            prompt_tokens=prompt_tokens,
            generated_tokens=output_tokens,
            ttft_ms=ttft,
            tbt_ms=tbt_median,
            e2e_ms=e2e,
            generation_text="[simulated]",
            quality_score=quality,
        )

# Test the simulator
chip = APPLE_CHIPS["M3 Max"]
model_7b_q4 = ModelConfig("Llama-3.2-7B-Q4", 7.0, 4, 8192)

sim = AppleSiliconSimulator(chip, model_7b_q4)
rng = np.random.default_rng(42)
test_rec = sim.simulate_request(256, 128, rng)
print(f"Test request on {chip.name} + {model_7b_q4.name}:")
print(f"  Model size: {model_7b_q4.model_size_gb:.1f} GB")
print(f"  TTFT:    {test_rec.ttft_ms:.1f} ms")
print(f"  TBT:     {test_rec.tbt_ms:.1f} ms/token")
print(f"  E2E:     {test_rec.e2e_ms:.0f} ms")
print(f"  Speed:   {test_rec.generated_tokens / (test_rec.e2e_ms/1000):.1f} tokens/sec")
print(f"  Quality: {test_rec.quality_score:.3f}")


In [ ]:
# ── Benchmark runner ─────────────────────────────────────────────────────────

class BenchmarkRunner:
    """
    Runs a suite of prompts against a simulator (or real Ollama if available),
    returns a BenchmarkRun with all measurements.
    """
    
    def __init__(self, n_warmup: int = 5, n_measure: int = 50, seed: int = 42):
        self.n_warmup = n_warmup
        self.n_measure = n_measure
        self.rng = np.random.default_rng(seed)
    
    def _generate_prompt_lengths(self, n: int):
        """Realistic prompt + output length distribution from production logs."""
        prompt_tokens = self.rng.lognormal(mean=5.5, sigma=0.7, size=n).astype(int).clip(50, 2048)
        output_tokens = self.rng.lognormal(mean=4.8, sigma=0.8, size=n).astype(int).clip(20, 512)
        return prompt_tokens, output_tokens
    
    def run_simulated(self, simulator: AppleSiliconSimulator,
                       model_name: str) -> BenchmarkRun:
        """Run benchmark using the Apple Silicon simulator."""
        prompt_lens, output_lens = self._generate_prompt_lengths(self.n_warmup + self.n_measure)
        
        # Warmup (discard)
        for i in range(self.n_warmup):
            simulator.simulate_request(int(prompt_lens[i]), int(output_lens[i]), self.rng)
        
        # Measurement
        records = []
        for i in range(self.n_warmup, self.n_warmup + self.n_measure):
            rec = simulator.simulate_request(int(prompt_lens[i]), int(output_lens[i]), self.rng)
            records.append(rec)
        
        return BenchmarkRun(
            model_name=model_name,
            records=records,
            system_info={
                "chip": simulator.chip.name,
                "bandwidth_gbs": simulator.chip.memory_bandwidth_gbs,
                "model": model_name,
                "quant_bits": simulator.model.quant_bits,
                "n_requests": self.n_measure,
            }
        )
    
    def run_ollama(self, model_name: str, prompts: list[str]) -> Optional[BenchmarkRun]:
        """Run against a real Ollama instance if available."""
        if not OLLAMA_AVAILABLE:
            return None
        try:
            import ollama as ol
            import time as t
            records = []
            for prompt in prompts[:self.n_measure]:
                t0 = t.perf_counter()
                first_token_time = None
                tokens = []
                stream = ol.chat(model=model_name,
                                 messages=[{"role": "user", "content": prompt}],
                                 stream=True)
                for chunk in stream:
                    if first_token_time is None:
                        first_token_time = t.perf_counter()
                    tokens.append(chunk.get("message", {}).get("content", ""))
                t1 = t.perf_counter()
                text = "".join(tokens)
                token_count = len(text.split())  # rough
                e2e_ms = (t1 - t0) * 1000
                ttft_ms = ((first_token_time or t1) - t0) * 1000
                records.append(RequestRecord(
                    prompt_tokens=len(prompt.split()),
                    generated_tokens=token_count,
                    ttft_ms=ttft_ms,
                    tbt_ms=(e2e_ms - ttft_ms) / max(token_count, 1),
                    e2e_ms=e2e_ms,
                    generation_text=text,
                    quality_score=0.0,  # would need eval harness
                ))
            return BenchmarkRun(model_name=model_name, records=records, system_info={})
        except Exception as e:
            print(f"Ollama error: {e}")
            return None

print("BenchmarkRunner defined.")
runner = BenchmarkRunner(n_warmup=5, n_measure=50)


## Bootstrap Confidence Intervals

The core statistical engine. For any metric function, we compute:
- Point estimate (sample statistic)
- 95% bootstrap CI using the **percentile method**
- Standard error

In [ ]:
# ── Bootstrap confidence interval engine ─────────────────────────────────────

def bootstrap_ci(data: np.ndarray,
                 statistic_fn: Callable,
                 n_bootstrap: int = 2000,
                 confidence: float = 0.95,
                 seed: int = 42) -> tuple[float, float, float]:
    """
    Compute bootstrap CI for statistic_fn applied to data.
    Returns (estimate, lower_bound, upper_bound).
    """
    rng = np.random.default_rng(seed)
    estimate = statistic_fn(data)
    boots = np.array([
        statistic_fn(rng.choice(data, size=len(data), replace=True))
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - confidence) / 2
    lo = np.percentile(boots, alpha * 100)
    hi = np.percentile(boots, (1 - alpha) * 100)
    return float(estimate), float(lo), float(hi)

def bootstrap_diff_ci(data_a: np.ndarray, data_b: np.ndarray,
                      statistic_fn: Callable,
                      n_bootstrap: int = 2000,
                      confidence: float = 0.95,
                      seed: int = 42) -> tuple[float, float, float, bool]:
    """
    Bootstrap CI on the difference statistic_fn(A) - statistic_fn(B).
    Returns (diff_estimate, lo, hi, is_significant).
    Significant = CI excludes 0.
    """
    rng = np.random.default_rng(seed)
    diff_est = statistic_fn(data_a) - statistic_fn(data_b)
    boot_diffs = []
    for _ in range(n_bootstrap):
        boot_a = rng.choice(data_a, size=len(data_a), replace=True)
        boot_b = rng.choice(data_b, size=len(data_b), replace=True)
        boot_diffs.append(statistic_fn(boot_a) - statistic_fn(boot_b))
    boot_diffs = np.array(boot_diffs)
    alpha = (1 - confidence) / 2
    lo = np.percentile(boot_diffs, alpha * 100)
    hi = np.percentile(boot_diffs, (1 - alpha) * 100)
    significant = not (lo <= 0 <= hi)
    return float(diff_est), float(lo), float(hi), significant

# Demo: sanity check with known distribution
x = np.random.default_rng(0).exponential(scale=50, size=100)  # ~Exp(50ms) TTFT
est, lo, hi = bootstrap_ci(x, np.median)
print(f"Sample median: {np.median(x):.2f}ms  95% CI: [{lo:.2f}, {hi:.2f}]")
print(f"True median:   {50 * np.log(2):.2f}ms  (Exp distribution)")
print()
# Two-sample difference
y = np.random.default_rng(1).exponential(scale=45, size=100)
diff, dlo, dhi, sig = bootstrap_diff_ci(x, y, np.median)
print(f"Diff (x-y) median: {diff:.2f}ms  95% CI: [{dlo:.2f}, {dhi:.2f}]")
print(f"Significant (CI excludes 0): {sig}")


In [ ]:
# ── Multi-model benchmark ─────────────────────────────────────────────────────

# Define model configurations to benchmark
chip = APPLE_CHIPS["M3 Max"]

MODEL_CONFIGS = [
    ModelConfig("Llama-3.2-3B-Q4",  3.0,  4, 8192),
    ModelConfig("Llama-3.2-7B-Q4",  7.0,  4, 8192),
    ModelConfig("Llama-3.2-7B-Q8",  7.0,  8, 8192),
    ModelConfig("Llama-3.1-8B-Q4",  8.0,  4, 131072),
    ModelConfig("Mistral-7B-Q4",    7.0,  4, 32768),
    ModelConfig("Llama-3.1-8B-Q8",  8.0,  8, 131072),
]

runs = {}
for mc in MODEL_CONFIGS:
    try:
        sim = AppleSiliconSimulator(chip, mc, efficiency=0.72)
        run = runner.run_simulated(sim, mc.name)
        runs[mc.name] = run
        tps_median = np.median(run.tokens_per_second())
        ttft_median = np.median(run.ttft_ms())
        print(f"  {mc.name:<25} {tps_median:>6.1f} tok/s  TTFT {ttft_median:>5.0f}ms  "
              f"size {mc.model_size_gb:>5.1f}GB  q{mc.quant_bits}")
    except ValueError as e:
        print(f"  {mc.name:<25} SKIP: {e}")

print(f"\nBenchmarked {len(runs)} models on simulated {chip.name}")


In [ ]:
# ── Results table with bootstrap CIs ─────────────────────────────────────────

def summarize_run(run: BenchmarkRun, n_bootstrap: int = 1000) -> dict:
    tps = run.tokens_per_second()
    ttft = run.ttft_ms()
    tbt = run.tbt_ms()
    qual = run.quality()
    
    tps_est, tps_lo, tps_hi = bootstrap_ci(tps, np.median, n_bootstrap)
    ttft_est, ttft_lo, ttft_hi = bootstrap_ci(ttft, np.median, n_bootstrap)
    qual_est, qual_lo, qual_hi = bootstrap_ci(qual, np.mean, n_bootstrap)
    
    return {
        "model": run.model_name,
        "tps_median": tps_est,
        "tps_ci": (tps_lo, tps_hi),
        "ttft_median_ms": ttft_est,
        "ttft_ci": (ttft_lo, ttft_hi),
        "quality_mean": qual_est,
        "quality_ci": (qual_lo, qual_hi),
        "n": run.n,
    }

summaries = {name: summarize_run(run) for name, run in runs.items()}

# Print formatted table
print(f"{'Model':<26} {'Tok/s (median)':>20} {'TTFT ms (median)':>20} {'Quality (mean)':>18}")
print(f"{'':26} {'[95% CI]':>20} {'[95% CI]':>20} {'[95% CI]':>18}")
print("-" * 88)
for name, s in summaries.items():
    tci = s['tps_ci']
    fci = s['ttft_ci']
    qci = s['quality_ci']
    print(f"{s['model']:<26} "
          f"{s['tps_median']:>6.1f} [{tci[0]:>5.1f},{tci[1]:>5.1f}] "
          f"{s['ttft_median_ms']:>6.0f} [{fci[0]:>5.0f},{fci[1]:>5.0f}] "
          f"{s['quality_mean']:>6.3f} [{qci[0]:>5.3f},{qci[1]:>5.3f}]")


In [ ]:
# ── Pairwise significance testing ────────────────────────────────────────────

print("Pairwise significance tests: tokens/sec (median)")
print("Positive diff = row model is FASTER than column model")
print()
model_names = list(runs.keys())

# Build significance matrix
sig_matrix = {}
for name_a in model_names:
    sig_matrix[name_a] = {}
    for name_b in model_names:
        if name_a == name_b:
            sig_matrix[name_a][name_b] = None
            continue
        diff, lo, hi, sig = bootstrap_diff_ci(
            runs[name_a].tokens_per_second(),
            runs[name_b].tokens_per_second(),
            np.median, n_bootstrap=1000
        )
        sig_matrix[name_a][name_b] = (diff, lo, hi, sig)

# Print matrix (abbreviated)
short_names = {n: n.split("-Q")[0].replace("Llama-3.", "L3.").replace("Mistral", "Mis") 
               for n in model_names}
header_names = [short_names[n][:12] for n in model_names]
print(f"{'':>18}", end="")
for h in header_names:
    print(f" {h:>13}", end="")
print()
print("-" * (18 + 14 * len(model_names)))

for name_a in model_names:
    print(f"{short_names[name_a]:>18}", end="")
    for name_b in model_names:
        if name_a == name_b:
            print(f" {'---':>13}", end="")
        else:
            diff, lo, hi, sig = sig_matrix[name_a][name_b]
            marker = "**" if sig else "  "
            print(f" {diff:>+7.1f}{marker}    ", end="")
    print()
print()
print("** = statistically significant at 95% confidence")


In [ ]:
# ── Benchmark visualization dashboard ────────────────────────────────────────

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle(f"benchpress: Multi-Model Comparison ({chip.name})", 
             fontsize=14, fontweight='bold')

colors = plt.cm.tab10(np.linspace(0, 1, len(runs)))
model_names = list(runs.keys())

# ─ Plot 1: Tokens/sec with CIs (forest plot) ─────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
y_pos = np.arange(len(model_names))
tps_vals = [summaries[n]["tps_median"] for n in model_names]
tps_los  = [summaries[n]["tps_median"] - summaries[n]["tps_ci"][0] for n in model_names]
tps_his  = [summaries[n]["tps_ci"][1] - summaries[n]["tps_median"] for n in model_names]

ax1.barh(y_pos, tps_vals, xerr=[tps_los, tps_his], color=colors, alpha=0.8,
         capsize=4, error_kw={"elinewidth": 1.5})
ax1.set_yticks(y_pos)
ax1.set_yticklabels([n.replace("Llama-3.", "L3.") for n in model_names], fontsize=8)
ax1.set_xlabel("Median tok/s (95% CI)")
ax1.set_title("Throughput (higher = better)")
ax1.grid(axis='x', alpha=0.3)
ax1.axvline(0, color='black', lw=0.5)

# ─ Plot 2: TTFT with CIs ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ttft_vals = [summaries[n]["ttft_median_ms"] for n in model_names]
ttft_los  = [summaries[n]["ttft_median_ms"] - summaries[n]["ttft_ci"][0] for n in model_names]
ttft_his  = [summaries[n]["ttft_ci"][1] - summaries[n]["ttft_median_ms"] for n in model_names]

ax2.barh(y_pos, ttft_vals, xerr=[ttft_los, ttft_his], color=colors, alpha=0.8,
         capsize=4, error_kw={"elinewidth": 1.5})
ax2.set_yticks(y_pos)
ax2.set_yticklabels([n.replace("Llama-3.", "L3.") for n in model_names], fontsize=8)
ax2.set_xlabel("Median TTFT ms (95% CI)")
ax2.set_title("Time-to-First-Token (lower = better)")
ax2.grid(axis='x', alpha=0.3)

# ─ Plot 3: Quality with CIs ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
qual_vals = [summaries[n]["quality_mean"] for n in model_names]
qual_los  = [summaries[n]["quality_mean"] - summaries[n]["quality_ci"][0] for n in model_names]
qual_his  = [summaries[n]["quality_ci"][1] - summaries[n]["quality_mean"] for n in model_names]

ax3.barh(y_pos, qual_vals, xerr=[qual_los, qual_his], color=colors, alpha=0.8,
         capsize=4, error_kw={"elinewidth": 1.5})
ax3.set_yticks(y_pos)
ax3.set_yticklabels([n.replace("Llama-3.", "L3.") for n in model_names], fontsize=8)
ax3.set_xlabel("Mean quality (95% CI)")
ax3.set_title("Quality Score (higher = better)")
ax3.grid(axis='x', alpha=0.3)
ax3.set_xlim(0.5, 1.05)

# ─ Plot 4: Throughput distribution (violin) ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
tps_data = [runs[n].tokens_per_second() for n in model_names]
vp = ax4.violinplot(tps_data, positions=np.arange(len(model_names)),
                    showmedians=True, showextrema=True)
for i, body in enumerate(vp['bodies']):
    body.set_facecolor(colors[i])
    body.set_alpha(0.7)
ax4.set_xticks(np.arange(len(model_names)))
ax4.set_xticklabels([n.replace("Llama-3.", "L3.").replace("-", "
") for n in model_names],
                     fontsize=7)
ax4.set_ylabel("Tokens/second")
ax4.set_title("Throughput Distribution (violin)")
ax4.grid(axis='y', alpha=0.3)

# ─ Plot 5: Speed vs Quality scatter ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
for i, name in enumerate(model_names):
    tps = summaries[name]["tps_median"]
    qual = summaries[name]["quality_mean"]
    ax5.scatter(tps, qual, color=colors[i], s=80, zorder=5)
    short = name.replace("Llama-3.", "L3.").replace("Mistral", "Mis")
    ax5.annotate(short, (tps, qual), fontsize=6, xytext=(3, 3),
                 textcoords='offset points')
ax5.set_xlabel("Median tokens/sec")
ax5.set_ylabel("Mean quality")
ax5.set_title("Speed-Quality Pareto")
ax5.grid(alpha=0.3)

plt.savefig('20_benchmark_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved 20_benchmark_dashboard.png")


In [ ]:
# ── Composite benchpress score ───────────────────────────────────────────────
# A single number combining speed and quality with configurable weights.
# Normalized to [0, 1] so scores are comparable across hardware.

def benchpress_score(run: BenchmarkRun, 
                     weight_speed: float = 0.5,
                     weight_quality: float = 0.5,
                     baseline_tps: float = 30.0,   # normalization baseline
                     baseline_qual: float = 0.70,
                     n_bootstrap: int = 1000) -> dict:
    """
    Composite score = w_s * (tps / baseline_tps) + w_q * (quality / baseline_qual)
    Both components > 1 means beating the baseline.
    Bootstrap CI computed on the composite score.
    """
    assert abs(weight_speed + weight_quality - 1.0) < 1e-9, "Weights must sum to 1"
    
    def composite(indices_or_data):
        if len(indices_or_data) == run.n:
            tps_arr = run.tokens_per_second()[indices_or_data] if indices_or_data.dtype == int else indices_or_data[:, 0]
        # Use paired bootstrap (resample request indices)
        tps_arr = run.tokens_per_second()
        qual_arr = run.quality()
        return (weight_speed * np.median(tps_arr) / baseline_tps +
                weight_quality * np.mean(qual_arr) / baseline_qual)
    
    # Paired bootstrap on request indices
    rng = np.random.default_rng(42)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, run.n, size=run.n)
        tps_b = run.tokens_per_second()[idx]
        qual_b = run.quality()[idx]
        scores.append(weight_speed * np.median(tps_b) / baseline_tps +
                      weight_quality * np.mean(qual_b) / baseline_qual)
    
    scores = np.array(scores)
    point_est = (weight_speed * np.median(run.tokens_per_second()) / baseline_tps +
                 weight_quality * np.mean(run.quality()) / baseline_qual)
    
    return {
        "model": run.model_name,
        "score": point_est,
        "score_ci": (float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))),
        "rank_stability": float(np.std(scores)),  # lower = more stable ranking
    }

print(f"benchpress Composite Scores (50% speed / 50% quality)")
print(f"Baseline: {30:.0f} tok/s, quality {0.70:.2f}")
print()
print(f"{'Model':<26} {'Score':>8} {'95% CI':>18} {'Rank Stability':>16}")
print("-" * 72)
bp_scores = [benchpress_score(runs[n]) for n in model_names]
bp_scores.sort(key=lambda x: x["score"], reverse=True)
for i, s in enumerate(bp_scores, 1):
    ci = s["score_ci"]
    print(f"  {i}. {s['model']:<23} {s['score']:>7.3f}  [{ci[0]:>5.3f}, {ci[1]:>5.3f}]  "
          f"±{s['rank_stability']:>6.4f}")

print()
print("Score > 1.0 = beats baseline on weighted metric")


## Apple Silicon Memory Bandwidth Analysis

On Apple Silicon, the roofline model tells us which models are memory-bound vs compute-bound.

The **roofline model** plots achievable performance against arithmetic intensity (FLOPs/byte):
- Models below the bandwidth roof: **memory-bound** (most LLM decode)
- Models above the compute roof: **compute-bound** (prefill with long sequences, training)

Understanding this helps explain why:
- Larger models saturate bandwidth faster
- Q4 quantization is disproportionately beneficial (4× smaller → 4× faster decode)
- Batch size > 1 helps prefill but barely helps decode

In [ ]:
# ── Memory bandwidth roofline analysis ───────────────────────────────────────

def bandwidth_utilization(chip: AppleSiliconChip, model: ModelConfig,
                           efficiency: float = 0.72) -> dict:
    """
    Analyze how much of peak bandwidth is being used for decode.
    """
    model_bytes = model.model_size_gb * 1e9
    bandwidth = chip.memory_bandwidth_gbs * 1e9  # bytes/sec
    
    # Theoretical max tokens/sec at perfect efficiency
    theoretical_max_tps = bandwidth / model_bytes
    # Achieved (with efficiency factor)
    achieved_tps = theoretical_max_tps * efficiency
    # Bandwidth utilization (decode)
    bw_utilization = efficiency  # by definition for memory-bound decode
    
    return {
        "theoretical_max_tps": theoretical_max_tps,
        "achieved_tps": achieved_tps,
        "bw_utilization_pct": bw_utilization * 100,
        "model_size_gb": model.model_size_gb,
    }

# Analyze all models on M3 Max
print(f"Memory Bandwidth Analysis: {chip.name} ({chip.memory_bandwidth_gbs} GB/s)")
print()
print(f"{'Model':<26} {'Size (GB)':>10} {'Theor. max':>12} {'Achieved':>10} {'BW util':>10}")
print(f"{'':26} {'':10} {'tok/s':>12} {'tok/s':>10} {'':10}")
print("-" * 72)

chip_test = APPLE_CHIPS["M3 Max"]
for mc in MODEL_CONFIGS:
    try:
        AppleSiliconSimulator(chip_test, mc)  # check if it fits
        bw = bandwidth_utilization(chip_test, mc)
        print(f"{mc.name:<26} {bw['model_size_gb']:>9.1f}  "
              f"{bw['theoretical_max_tps']:>11.1f}  "
              f"{bw['achieved_tps']:>9.1f}  "
              f"{bw['bw_utilization_pct']:>8.0f}%")
    except ValueError:
        print(f"{mc.name:<26} SKIP (doesn't fit)")

print()
# Compare across chips for 7B Q4
print("7B Q4 throughput across Apple Silicon chips:")
print(f"{'Chip':<15} {'BW (GB/s)':>10} {'Theor. tok/s':>14} {'Achiev. tok/s':>14}")
print("-" * 55)
for chip_name, chip_obj in APPLE_CHIPS.items():
    mc = ModelConfig("Llama-7B-Q4", 7.0, 4, 8192)
    if mc.model_size_gb <= chip_obj.max_ram_gb * 0.85:
        bw = bandwidth_utilization(chip_obj, mc)
        print(f"{chip_name:<15} {chip_obj.memory_bandwidth_gbs:>10.1f}  "
              f"{bw['theoretical_max_tps']:>14.1f}  {bw['achieved_tps']:>14.1f}")


In [ ]:
# ── Reproducibility & run metadata ───────────────────────────────────────────
# A real benchpress run captures system state alongside measurements.
# Without this, the same numbers mean nothing 6 months later.

import platform
from datetime import datetime

def capture_system_info() -> dict:
    """
    Capture reproducibility metadata for a benchmark run.
    In a real tool: also captures GPU/memory via psutil, model hash, etc.
    """
    return {
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python_version": sys.version.split()[0],
        "numpy_version": np.__version__,
        # In production: chip_name, memory_gb, thermal_state, ollama_version, model_sha256
        "note": "Simulated run — attach real hardware info when running on device"
    }

def save_benchmark_results(runs: dict, summaries: dict, bp_scores: list,
                             filename: str = "20_benchpress_results.json"):
    """Serialize benchmark results to JSON for later analysis."""
    output = {
        "metadata": capture_system_info(),
        "chip": chip.name,
        "models": {},
    }
    for name, run in runs.items():
        s = summaries[name]
        output["models"][name] = {
            "tps_median": s["tps_median"],
            "tps_ci_lo": s["tps_ci"][0],
            "tps_ci_hi": s["tps_ci"][1],
            "ttft_median_ms": s["ttft_median_ms"],
            "ttft_ci_lo": s["ttft_ci"][0],
            "ttft_ci_hi": s["ttft_ci"][1],
            "quality_mean": s["quality_mean"],
            "quality_ci_lo": s["quality_ci"][0],
            "quality_ci_hi": s["quality_ci"][1],
            "n_requests": run.n,
            "raw_tps": run.tokens_per_second().tolist(),
            "raw_ttft": run.ttft_ms().tolist(),
            "raw_quality": run.quality().tolist(),
        }
    output["benchpress_scores"] = bp_scores
    
    import json
    with open(filename, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Results saved to {filename}")
    return output

results = save_benchmark_results(runs, summaries, bp_scores)
print(f"\nSystem info captured:")
for k, v in results["metadata"].items():
    print(f"  {k}: {v}")


## Series Retrospective

Twenty notebooks from tokenization to benchmarking. Here's the thread that runs through all of them:

### The Core Stack (How a Token Gets Generated)

```
User prompt
    ↓ tokenize (nb 01: BPE, WordPiece)
    ↓ embed + positional encode (nb 16: RoPE, base frequency)
    ↓ transformer layers (nb 02: attention, MLP, LayerNorm)
    ↓ KV cache lookup / populate (nb 05, 07: PagedAttention)
    ↓ flash attention kernel (nb 06: tiled compute)
    ↓ prefix cache hit? (nb 08: reuse saved KV blocks)
    ↓ sample from logits (nb 03: temperature, top-p, top-k)
    ↓ constrained sampling? (nb 15: logit masking, JSON FSA)
    ↓ speculative acceptance? (nb 10: draft-verify loop)
    ↓ continuous batch scheduler (nb 11: Orca iteration-level)
    ↓ vLLM engine step (nb 14: BlockManager, Scheduler)
    ↓ measure TTFT / TBT / throughput (nb 12: serving metrics)
    ↓ run on Apple Silicon (nb 13: Ollama, MLX, bandwidth model)
    ↓ parallelism strategy (nb 18: TP + PP)
    ↓ context extension if needed (nb 17: YaRN, PI, higher base)
    ↓ quantize if memory-constrained (nb 09: GGUF Q4)
    ↓ route to right-sized model (nb 19: UCB router)
    ↓ benchmark with statistical rigor (nb 20: benchpress)
Token output
```

### What `benchpress` Needs to Graduate from Prototype

| Feature | This notebook | Production `benchpress` |
|---------|--------------|------------------------|
| Timing source | Simulator | Real Ollama / MLX streaming API |
| Quality metric | Simulated score | lm-eval-harness task accuracy |
| Hardware profiling | Bandwidth model | psutil + powermetrics for thermal |
| Statistical tests | Bootstrap percentile | Bootstrap + Welch's t-test + BH correction |
| Reproducibility | Basic metadata | Model SHA256, thermal state, run config hash |
| Output format | JSON | JSON + Markdown report + PNG charts |
| Comparisons | 6 configs | Any Ollama-pullable model |

### The Backlog Connection

Every tool in the backlog is now approachable because of this series:

| Backlog Tool | Foundation Notebooks |
|--------------|---------------------|
| `benchpress` | 12 (metrics), 13 (local), 20 (capstone) |
| `dispatch` | 19 (routing), 12 (metrics) |
| `lossy` | 09 (quantization), 12 (metrics), 20 (bootstrap CIs) |
| `drift` | 12 (distribution metrics), 20 (bootstrap diff CI) |

## Summary

In this capstone we built a complete benchmarking pipeline:

| Component | Implementation | Key Concept |
|-----------|---------------|-------------|
| **Timing simulator** | `AppleSiliconSimulator` | Bandwidth model: TBT = model_bytes / bandwidth |
| **Benchmark runner** | `BenchmarkRunner` | Warmup + measure; realistic prompt distribution |
| **Bootstrap CIs** | `bootstrap_ci`, `bootstrap_diff_ci` | Percentile method; works for any statistic |
| **Results table** | `summarize_run` | Every number has a confidence interval |
| **Significance tests** | Paired bootstrap difference CI | "Faster" only if CI on diff excludes 0 |
| **Composite score** | `benchpress_score` | Weighted speed + quality; paired bootstrap CI |
| **Bandwidth analysis** | `bandwidth_utilization`, roofline | Decode is always memory-bound on Apple Silicon |
| **Reproducibility** | `capture_system_info`, JSON output | Benchmark results without metadata are useless |

### The Three Numbers Every Benchmark Should Report

1. **Point estimate** — the median / mean (obvious, but often all that gets reported)
2. **95% bootstrap CI** — how much would this change if we ran it again?
3. **Significance test** — is the difference between Model A and B real, or noise?

Without all three, you're doing a vibes check with extra steps.

---

*Series complete. 20 notebooks, one complete mental model of LLM inference from tokens to serving.*